## Take-home Exercise 3
### Haoxuan Lu
### CUID: hl3921
### Date: Apr.14, 2026

### Homework #4: model building and tracking
Instructions: For this assignment, we’d like you to use the F1 Datasets we have been using for the class to build any ML model of your choice and track the model for each run using MLflow. Select any of the F1 datasets in AWS S3 to build your model. You are allowed to join multiple datasets.

##### 0. Basic Setup

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/gr5069/raw/f1_data"))
     
df = spark.read.csv(
    "dbfs:/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

display(df)

In [0]:
df_pd = df.toPandas()
df_pd = df_pd.replace("\\N", np.nan)

print(df_pd.columns.tolist())
print(df_pd.head())

In [0]:
selected_cols = [
    "grid",
    "laps",
    "fastestLap",
    "rank",
    "fastestLapSpeed",
    "points",
    "positionOrder"
]

df_model = df_pd[selected_cols].copy()

for col in df_model.columns:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

df_model = df_model.dropna()

X = df_model.drop("positionOrder", axis=1)
y = df_model["positionOrder"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
display(df_model.head())
     

In [0]:
import mlflow
mlflow.set_experiment("/Shared/F1-MLflow-Experiment")

##### Question 1 — Build any model of your choice with tunable hyperparameters

I chose a Random Forest Regressor to predict `positionOrder` using race-result-related numeric features from the F1 `results.csv` dataset. I selected this model because it supports several tunable hyperparameters, can capture nonlinear relationships, and works well without strict linearity assumptions. The main hyperparameters tuned in this assignment are `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, and `max_features`.

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score,
    max_error
)
import matplotlib.pyplot as plt
import os

In [0]:

rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)
medae = median_absolute_error(y_test, y_pred)
evs = explained_variance_score(y_test, y_pred)
maxerr = max_error(y_test, y_pred)

print("Single baseline Random Forest model results:")
print(f"MAE:   {mae:.4f}")
print(f"MSE:   {mse:.4f}")
print(f"RMSE:  {rmse:.4f}")
print(f"R2:    {r2:.4f}")
print(f"MedAE: {medae:.4f}")
print(f"EVS:   {evs:.4f}")
print(f"Max Error: {maxerr:.4f}")

##### Question 2 — Create an experiment setup and log params, model, metrics, and at least two artifacts

In [0]:
import mlflow
import mlflow.sklearn
import tempfile
import pandas as pd

In [0]:
def run_rf_experiment(params, run_name=None):
    """
    Train one RandomForestRegressor model and log:
    - hyperparameters
    - model
    - evaluation metrics
    - artifacts: feature importance plot, actual vs predicted plot, predictions csv
    """
    
    with mlflow.start_run(run_name=run_name):
        # Build model
        model = RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=42,
            n_jobs=-1
        )
        
        # Train
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        # Metrics
        mae = mean_absolute_error(y_test, preds)
        mse = mean_squared_error(y_test, preds)
        rmse = mse ** 0.5
        r2 = r2_score(y_test, preds)
        medae = median_absolute_error(y_test, preds)
        evs = explained_variance_score(y_test, preds)
        maxerr = max_error(y_test, preds)
        
        # Log params
        mlflow.log_params(params)
        
        # Log metrics
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("median_absolute_error", medae)
        mlflow.log_metric("explained_variance", evs)
        mlflow.log_metric("max_error", maxerr)
        
        # Log model
        mlflow.sklearn.log_model(model, "random_forest_model")
        
        # Create temp folder for artifacts
        with tempfile.TemporaryDirectory() as tmpdir:
            
            # Artifact 1: predictions CSV
            pred_df = pd.DataFrame({
                "actual_positionOrder": y_test.values,
                "predicted_positionOrder": preds
            })
            pred_csv_path = os.path.join(tmpdir, "predictions.csv")
            pred_df.to_csv(pred_csv_path, index=False)
            mlflow.log_artifact(pred_csv_path, artifact_path="artifacts")
            
            # Artifact 2: actual vs predicted scatter plot
            plt.figure(figsize=(7, 5))
            plt.scatter(y_test, preds, alpha=0.5)
            plt.xlabel("Actual positionOrder")
            plt.ylabel("Predicted positionOrder")
            plt.title("Actual vs Predicted")
            scatter_path = os.path.join(tmpdir, "actual_vs_predicted.png")
            plt.savefig(scatter_path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(scatter_path, artifact_path="artifacts")
            
            # Artifact 3: feature importance plot
            importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
            plt.figure(figsize=(8, 5))
            importances.plot(kind="bar")
            plt.title("Feature Importance")
            plt.ylabel("Importance Score")
            fi_path = os.path.join(tmpdir, "feature_importance.png")
            plt.savefig(fi_path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(fi_path, artifact_path="artifacts")
        
        print("Run completed.")
        print("Parameters:", params)
        print(f"R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

In [0]:
# Test one MLflow run first
sample_params = {
    "n_estimators": 150,
    "max_depth": 10,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt"
}

run_rf_experiment(sample_params, run_name="rf_test_run")

For each MLflow run, I logged the model hyperparameters, the trained Random Forest model itself, and all evaluation metrics available from my regression setup, including MAE, MSE, RMSE, R², median absolute error, explained variance, and max error. I also logged at least two artifacts for every run: a predictions CSV file, an actual-vs-predicted scatter plot, and a feature-importance plot.

##### Question 3 — Track your MLflow experiment and run at least 10 experiments with different parameters each

In [0]:
param_grid = [
    {"n_estimators": 100, "max_depth": 8,  "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 150, "max_depth": 8,  "min_samples_split": 2, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": 8,  "min_samples_split": 5, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 5, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 150, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "log2"},
    {"n_estimators": 200, "max_depth": 10, "min_samples_split": 5, "min_samples_leaf": 2, "max_features": "log2"},
    {"n_estimators": 100, "max_depth": 12, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": None},
    {"n_estimators": 150, "max_depth": 12, "min_samples_split": 5, "min_samples_leaf": 1, "max_features": None},
    {"n_estimators": 200, "max_depth": 15, "min_samples_split": 2, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 250, "max_depth": 15, "min_samples_split": 5, "min_samples_leaf": 1, "max_features": "log2"}
]

print(f"Number of experiment runs: {len(param_grid)}")

In [0]:
for i, params in enumerate(param_grid, start=1):
    run_rf_experiment(params, run_name=f"rf_run_{i}")

In [0]:
# View experiment runs in a table
experiment = mlflow.get_experiment_by_name("/Shared/F1-MLflow-Experiment")
experiment_id = experiment.experiment_id

runs_df = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.r2 DESC"]
)

display(runs_df)

I ran 10 MLflow experiments with different combinations of hyperparameters to compare model performance. This allowed me to systematically evaluate how changes in model complexity and split settings affected prediction accuracy and error. All runs were tracked under the same MLflow experiment so they could be compared directly.

##### Question 4 — Select your best model run and explain why

In [0]:
best_run = runs_df.iloc[0]

print("Best run summary:")
print("Run ID:", best_run["run_id"])
print("R2:", best_run["metrics.r2"])
print("RMSE:", best_run["metrics.rmse"])
print("MAE:", best_run["metrics.mae"])
print("MSE:", best_run["metrics.mse"])
print("Median Absolute Error:", best_run["metrics.median_absolute_error"])
print("Explained Variance:", best_run["metrics.explained_variance"])
print("Max Error:", best_run["metrics.max_error"])

print("\nBest hyperparameters:")
print("n_estimators:", best_run["params.n_estimators"])
print("max_depth:", best_run["params.max_depth"])
print("min_samples_split:", best_run["params.min_samples_split"])
print("min_samples_leaf:", best_run["params.min_samples_leaf"])
print("max_features:", best_run["params.max_features"])

In [0]:
# show only the most relevant columns for reporting
best_model_table = runs_df[[
    "run_id",
    "metrics.r2",
    "metrics.rmse",
    "metrics.mae",
    "metrics.mse",
    "metrics.median_absolute_error",
    "metrics.explained_variance",
    "metrics.max_error",
    "params.n_estimators",
    "params.max_depth",
    "params.min_samples_split",
    "params.min_samples_leaf",
    "params.max_features"
]]

display(best_model_table)

I selected the best model run based primarily on the highest R² score, while also checking that RMSE and MAE were relatively low. I considered this run the best because it explained the greatest proportion of variance in `positionOrder` while keeping prediction error comparatively small. This makes it the strongest overall tradeoff between fit and prediction accuracy among the 10 experiments.